SOC612: Data Analytics for the Social Sciences - Solutions

Date: September 22, 2026

Author: R. Duerr

Task 2: Descriptive and exploratory analysis

**1.**


In [ ]:
import pandas as pd
import numpy as np

# --- Load Data ---
nhis22 = pd.read_csv("nhis-22.csv")

**2.**

In [ ]:
# Define the variables
variables = ['fries', 'potato', 'beans', 'pizza']
stats = []

# Calculate statistics for each variable
for var in variables:
    col = nhis22[var]
    mean_val = round(col.mean(), 2)
    median_val = round(col.median(), 2)
    sd_val = round(col.std(), 2)
    min_val = round(col.min(), 2)
    max_val = round(col.max(), 2)
    stats.append([mean_val, median_val, sd_val, min_val, max_val])

# Create the DataFrame (matching R's rbind structure)
variable = ["mean", "median", "sd", "min", "max"]
eating_habits = pd.DataFrame([
    variable,
    *stats
])

**3.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde

# --- Fruit Histograms ---
# 1: Histogram
plt.hist(nhis22['fruit'], bins=60, color='lightgreen', edgecolor='black', linewidth=2)
plt.title("Histogram")
plt.xlabel("Fruit")
plt.ylabel("Freq.")
plt.show()

# 2: ggplot2-style Histogram (Seaborn)
sns.histplot(data=nhis22, x='fruit', bins=60, color='lightgreen', edgecolor='black')
plt.title("Histogram")
plt.xlabel("Fruit")
plt.ylabel("Freq.")
plt.show()

# --- Veggie Density Plots ---
# Extract male and female veggie data
male_veggie = nhis22[nhis22['gender'] == 'male']['veggie']
female_veggie = nhis22[nhis22['gender'] == 'female']['veggie']

# 3: Density Plot
density_male = gaussian_kde(male_veggie)
density_female = gaussian_kde(female_veggie)
x = np.linspace(
    min(male_veggie.min(), female_veggie.min()),
    max(male_veggie.max(), female_veggie.max()),
    1000
)

plt.plot(x, density_male(x), color='orange', linewidth=2, label='Male')
plt.plot(x, density_female(x), color='purple', linewidth=2, label='Female')
plt.title("Density Plot of Veggie by Gender")
plt.xlabel("Veggie")
plt.ylabel("Dens.")
plt.xlim(min(male_veggie.min(), female_veggie.min()), max(male_veggie.max(), female_veggie.max()))
plt.ylim(0, max(density_male(x).max(), density_female(x).max()))
plt.legend(loc='upper right')
plt.show()

# 4: ggplot2-style Density Plot (Seaborn)
sns.kdeplot(data=nhis22, x='veggie', hue='gender', fill=True, alpha=0.5, palette=['orange', 'purple'])
plt.title("Density Plot of Veggie by Gender")
plt.xlabel("Veggie")
plt.ylabel("Dens.")
plt.show()

# --- Salad Boxplots ---
# 5: Boxplot
educ_levels = nhis22['educ'].unique()
salad_data = [nhis22[nhis22['educ'] == level]['salad'] for level in educ_levels]
colors = ['cyan', 'beige', 'grey', 'brown']

bp = plt.boxplot(salad_data, patch_artist=True)
for patch, color in zip(bp['boxes'], colors[:len(salad_data)]):
    patch.set_facecolor(color)
for whisker in bp['whiskers']:
    whisker.set_color('black')
for cap in bp['caps']:
    cap.set_color('black')
for median in bp['medians']:
    median.set_color('black')
plt.title("Boxplot of Veggie by Education")
plt.ylabel("Veggie")
plt.xticks(range(1, len(educ_levels) + 1), educ_levels)
plt.show()

# 6: ggplot2-style Boxplot (Seaborn)
sns.boxplot(data=nhis22, x='educ', y='veggie', hue='educ', palette=colors)
plt.title("Boxplot of Veggie by Education")
plt.xlabel("Education")
plt.ylabel("Veggie")
plt.show()

**4.**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Categorize coffee and soda consumption
for var in ['coffee', 'soda']:
    conditions = [
        (nhis22[var] == 0),
        (nhis22[var].between(1, 3)),
        (nhis22[var].between(4, 10)),
        (nhis22[var].between(11, 20)),
        (nhis22[var].between(21, 29)),
        (nhis22[var] == 30),
        (nhis22[var] >= 31)
    ]
    choices = ["never", "rarely", "sometimes", "regularly", "often", "daily", "daily+"]
    nhis22[f'{var}.cat'] = np.select(conditions, choices, default=np.nan)
    nhis22[f'{var}.cat'] = pd.Categorical(
        nhis22[f'{var}.cat'],
        categories=["never", "rarely", "sometimes", "regularly", "often", "daily", "daily+"],
        ordered=True
    )

# Bar Chart: Soda Consumption by Gender
sns.countplot(data=nhis22, x='soda.cat', hue='gender', palette=['lightblue', 'lightpink'])
plt.title("Bar Chart of Soda Consumption by Gender")
plt.xlabel("Soda")
plt.ylabel("Freq.")
plt.legend(title="Gender")
plt.show()

# Bar Chart: Coffee Consumption by Education
sns.countplot(data=nhis22, x='coffee.cat', hue='educ', palette=['lightblue', 'lightpink', 'lightgreen', 'purple'])
plt.title("Bar Chart of Coffee Consumption by Education")
plt.xlabel("Coffee")
plt.ylabel("Freq.")
plt.legend(title="Education")
plt.show()

**5.**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select columns for correlation
cols = ["soda", "fruit_juice_pure", "coffee", "sports_drink",
        "fruit_juice_sweet", "fruit", "salad", "fries", "potato",
        "beans", "veggie", "salsa", "pizza", "tomato_sauce"]
nhis22_cd = nhis22[cols]

# Calculate correlation matrix
corr_table = nhis22_cd.corr()

# Plot correlation matrix (ggcorrplot alternative)
plt.figure(figsize=(12, 10))
sns.heatmap(corr_table, annot=True, fmt=".2f", cmap='coolwarm', center=0, square=True)
plt.title("Correlation Matrix")
plt.show()

**6.**

In [ ]:
# Pizza
Q1p = nhis22['pizza'].quantile(0.25)
Q3p = nhis22['pizza'].quantile(0.75)
IQRp = Q3p - Q1p
lower_bound_pizza = Q1p - 1.5 * IQRp
upper_bound_pizza = Q3p + 1.5 * IQRp
nhis22['pizza_outliers'] = (nhis22['pizza'] < lower_bound_pizza) | (nhis22['pizza'] > upper_bound_pizza)

# Salsa
Q1s = nhis22['salsa'].quantile(0.25)
Q3s = nhis22['salsa'].quantile(0.75)
IQRs = Q3s - Q1s
lower_bound_salsa = Q1s - 1.5 * IQRs
upper_bound_salsa = Q3s + 1.5 * IQRs
nhis22['salsa_outliers'] = (nhis22['salsa'] < lower_bound_salsa) | (nhis22['salsa'] > upper_bound_salsa)

# Tomato Sauce
Q1t = nhis22['tomato_sauce'].quantile(0.25)
Q3t = nhis22['tomato_sauce'].quantile(0.75)
IQRt = Q3t - Q1t
lower_bound_tomsau = Q1t - 1.5 * IQRt
upper_bound_tomsau = Q3t + 1.5 * IQRt
nhis22['tomsau_outliers'] = (nhis22['tomato_sauce'] < lower_bound_tomsau) | (nhis22['tomato_sauce'] > upper_bound_tomsau)